# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready GeoParquet row per parcel with dated feature columns.

In [ ]:
import os
import sys
from pathlib import Path

from openeo_parcel_stats_pipeline.zonal_stats_multiuser_manager import (
    split_geodataframe_by_grid,
    load_openeo_users_from_env,
    run_parallel_extractions,
    merge_and_save_results,
)
from common_libraries.io_library import read_data
import common_libraries.geom_library as geom_l

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
WORK_PATH = Path("C:/work_dir/axios-2023-2024")
PARCELS_PATH = WORK_PATH / "parcels_axios.gpkg"

In [ ]:
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(len(gdf_parcels), "parcels in test area")
gdf_parcels.head(3)

In [ ]:
class_column = "initial_label_code"
parcel_id_column = "parcel_code"

In [ ]:
all_gdf = gdf_parcels.copy()
all_gdf["accuracy"] = all_gdf["initial_label_code"] == all_gdf["predicted_label_code"]
accuracy = all_gdf["accuracy"].mean()
print(f"Accuracy on all data: {accuracy:.2%}")


test_gdf = gdf_parcels[gdf_parcels["set_type"]=="test"].copy()
print(len(test_gdf), "rows in test_gdf after filtering with set_type=='test'")
test_gdf["accuracy"] = test_gdf["initial_label_code"] == test_gdf["predicted_label_code"]
accuracy = test_gdf["accuracy"].mean()
print(f"Accuracy on test data: {accuracy:.2%}")

In [ ]:
START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
RUN_IDENTIFIER = "axios-2023-2024"  # Change this for each named run.
OUTPUT_DIR = WORK_PATH / "1_parcel_stats"
WORKING_EPSG = 32634

CALCULATE_SENTINEL2_INDICES = True
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]
SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL1_BANDS = ["VV", "VH"]

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    # "count",
    "p10",
    "p25",
    "p75",
    "p90",
]

INTERPOLATION_METHOD = "nearest"  # "idw"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500

# IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True
IQR_MIN_VALID_PIXELS = 20
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 1

gdf_parcels = gdf_parcels.to_crs(WORKING_EPSG)

## Tile-size recommendation

This local comparison selects the smallest candidate grid whose buffered ownership tiles fully cover every parcel.

In [ ]:
TILE_SIZE_METRES = compute_tile_width(gdf_parcels, parcel_id_col=parcel_id_column, working_epsg=WORKING_EPSG)
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_parcels, WORKING_EPSG)

## Run extraction

In [ ]:
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = 2_100

In [ ]:
# ============================================================================
# STEP 1: Split parcels into spatial partitions
# ============================================================================
# This divides the parcels into a grid (1 row, 2 columns = left/right halves)
# For 5 users, use: rows=1, cols=5

spatial_parts = split_geodataframe_by_grid(gdf=gdf_parcels, rows=2, cols=2)

print(f"Created {len(spatial_parts)} spatial partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {len(part)} parcels")

In [ ]:
# ============================================================================
# STEP 2: Load openEO users from environment variable
# ============================================================================
# Requires OPENEO_USERS env var: "user1@example.com,pass1;user2@example.com,pass2"

users_list = load_openeo_users_from_env()

print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")


In [ ]:
# ============================================================================
# STEP 3: Create extraction configuration dictionary
# ============================================================================
# Single dictionary containing all parameters for the extraction process,
# to be passed to each parallel worker.

extraction_config = {
    # Output paths
    "output_dir": OUTPUT_DIR,
    "run_identifier": RUN_IDENTIFIER,

    # Parcel configuration
    "parcel_id_field": parcel_id_column,

    # Temporal bounds
    "start_date": START_DATE,
    "end_date": END_DATE,

    # Spatial configuration
    "working_epsg": WORKING_EPSG,

    # Sentinel-2 configuration
    "sentinel2_bands": SENTINEL2_BANDS,
    "calculate_sentinel2_indices": CALCULATE_SENTINEL2_INDICES,
    "sentinel2_indices": SENTINEL2_INDICES,

    # Sentinel-1 configuration
    "sentinel1_bands": SENTINEL1_BANDS,

    # Statistics to calculate
    "spatial_statistics": SPATIAL_STATISTICS,

    # Tiling configuration
    "tile_size_metres": TILE_SIZE_METRES,
    "tile_buffer_metres": TILE_BUFFER_METRES,

    # Data cleaning (IQR outlier removal)
    "remove_outliers": REMOVE_OUTLIERS,
    "iqr_quantiles": IQR_QUANTILES,
    "iqr_multiplier": IQR_MULTIPLIER,
    "iqr_min_valid_pixels": IQR_MIN_VALID_PIXELS,

    # Null filling
    "fill_nulls": FILL_NULLS,
    "temporal_fill_mode": TEMPORAL_FILL_MODE,
    "minimum_parcel_pixels": MINIMUM_PARCEL_PIXELS,
    "minimum_observed_fraction_for_fill": MINIMUM_OBSERVED_FRACTION_FOR_FILL,

    # Interpolation
    "interpolation_method": INTERPOLATION_METHOD,
    "interpolation_max_distance_in_meters": INTERPOLATION_MAX_DISTANCE_IN_METERS,

    # Parallel processing
    "openeo_parallel_jobs": 2,
    "job_poll_seconds": 30,
    "batch_workers": LOCAL_BATCH_WORKERS,

    # Other options
    "spatial_fill_window_sizes": (3, 5),
}

# ============================================================================
# STEP 4: Run all partitions in parallel
# ============================================================================
# Each partition uses a different user (round-robin assignment)
# All partitions extract simultaneously

all_results = run_parallel_extractions(
    spatial_parts=spatial_parts,
    users_list=users_list,
    config=extraction_config,
)

# Output format: {partition_idx: results_gdf}
print(f"\nExtraction complete!")
print(f"  Partition 1: {len(all_results[1])} rows")
print(f"  Partition 2: {len(all_results[2])} rows")

In [ ]:
# ============================================================================
# STEP 4: Merge results and save to files
# ============================================================================
# Combines all partition results and removes duplicates
# Saves two files:
#   1. satellite_parcel_time_stats.geoparquet (full results with geometry)
#   2. satellite_parcel_time_stats_preview.csv (100 rows + 10 median features)

merged_gdf, geoparquet_path = merge_and_save_results(
    gdf_parcels=gdf_parcels,
    all_results=all_results,
    output_dir=OUTPUT_DIR,
    parcel_id_column=parcel_id_column,
)

In [ ]:
# ============================================================================
# STEP 5: Inspect results
# ============================================================================

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")

print(f"\nMerged GeoDataFrame:")
print(f"  Rows: {len(merged_gdf)}")
print(f"  Columns: {len(merged_gdf.columns)}")
print(f"  Has geometry: {merged_gdf.geometry.name in merged_gdf.columns}")

print(f"\nOutput files:")
print(f"  GeoParquet: {geoparquet_path}")
print(f"    Size: {geoparquet_path.stat().st_size / 1e6:.1f} MB")

print(f"\nFirst 5 rows:")
print(merged_gdf.head())

print(f"\nColumn names (first 15):")
print(merged_gdf.columns[:15].tolist())